## Basic Setup

In [1]:
import os
import random
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from keras.layers import BatchNormalization
from keras.optimizers import SGD
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

os.chdir('../../../') # move three level up to the base pathfrom src.utils import load_and_preprocess_images
from src.utils import load_and_preprocess_images
from src.model_evaluation import evaluate_model, plot_history

In [2]:
# set the random seeds to make sure that the results are reproducible
SEED = 1234
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

In [3]:
model_name = '2_finetuned_real_eyes'
repo_path = '/Users/hendriksippel/Documents/Repositories/cbs-mldl-drowsiness-detection'

syntheticDataPath = repo_path + '/data/Unity_Data_Test_Train'
syntheticTrainingPath = syntheticDataPath + '/TrainingSet'
syntheticTestPath = syntheticDataPath + '/TestSet'

realDataPath = repo_path + '/data/CEW_Data_Test_Train'
realTrainingPath = realDataPath + '/TrainingSet'
realTestPath = realDataPath + '/TestSet'

In [5]:
comment = "midday run"
model_file_path = repo_path + f"/models/cnn/{model_name}/weights/{comment}_model.keras"
ckpt_file_path = repo_path + f"/models/cnn/{model_name}/ckpt/{comment}_checkpoint.model.keras"
history_file_path = repo_path + f"/models/cnn/{model_name}/history/{comment}_history.csv"
# assert not os.path.exists(ckpt_file_path), "Model already exists. Please change the comment."

In [6]:
# Hyperparameters
BATCH_SIZE = 64
IMAGE_SIZE = (224, 224)
INPUT_SHAPE = IMAGE_SIZE + (3,)

DATA_AUG_RATE = 0.1
LOSS_FUNCTION = 'sparse_categorical_crossentropy'

FT_EPOCHS = 100
FT_LEARNING_RATE = 0.00001
OPTIMIZER = SGD(learning_rate=FT_LEARNING_RATE)

In [7]:
early_stop = EarlyStopping(monitor="val_accuracy", mode="max", patience=20, start_from_epoch=10)
checkpoint = ModelCheckpoint(ckpt_file_path, monitor="val_accuracy", mode="max", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=10, min_lr=0.00001, verbose=1)

In [8]:
real_train_data = load_and_preprocess_images(realTrainingPath, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, seed=SEED, data_aug_rate=DATA_AUG_RATE, subset='training', validation_split=0.2)
real_valid_data = load_and_preprocess_images(realTrainingPath, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, seed=SEED, subset='validation', validation_split=0.2)
real_test_data = load_and_preprocess_images(realTestPath, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, seed=SEED, shuffle=False)

Found 3876 files belonging to 2 classes.
Using 3101 files for training.
Found 3876 files belonging to 2 classes.
Using 775 files for validation.
Found 970 files belonging to 2 classes.


## Finetuning the Pre-Trained Model

In [9]:
# define the path to the pretrained model
pretrained_model_path = repo_path + '/models/cnn/1_synthetic_eyes/1_full_network/weights/evening run_model.keras'

# load the model with compiling
model = tf.keras.models.load_model(pretrained_model_path, compile=True)

# get benchmark results by evaluating the model on the test set
evaluate_model(model, real_test_data)

16/16 ━━━━━━━━━━━━━━━━━━━━ 12s 621ms/step - accuracy: 0.2855 - loss: 9.1449
Test loss: 8.420035362243652
Test accuracy: 0.47010308504104614


In [ ]:
# load and override the model again without compiling it
model = tf.keras.models.load_model(pretrained_model_path, compile=False)

# unfreeze all layers which are not BN layers
for layer in model.layers:
    if isinstance(layer, BatchNormalization): # freeze BN layers
        layer.trainable = False
    else: # unfreeze all other layers
        layer.trainable = True

# display the summary if needed
# model.summary(show_trainable=True)

In [10]:
# compile the model with the same optimizer and loss function
model.compile(loss=LOSS_FUNCTION, optimizer=OPTIMIZER, metrics=['accuracy'])

# train the entire model with the same data but lower learning rate
history = model.fit(
    real_train_data,
    validation_data=real_valid_data,
    epochs=FT_EPOCHS,
    callbacks=[checkpoint, reduce_lr] # early_stop
    )

Epoch 1/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6063 - loss: 7.8785
Epoch 1: val_accuracy improved from -inf to 0.79742, saving model to /Users/hendriksippel/Documents/Repositories/cbs-mldl-drowsiness-detection/models/cnn/2_finetuned_real_eyes/ckpt/midday run_checkpoint.model.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step - accuracy: 0.6075 - loss: 7.8726 - val_accuracy: 0.7974 - val_loss: 7.1118 - learning_rate: 1.0000e-05
Epoch 2/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7559 - loss: 7.2521
Epoch 2: val_accuracy improved from 0.79742 to 0.87097, saving model to /Users/hendriksippel/Documents/Repositories/cbs-mldl-drowsiness-detection/models/cnn/2_finetuned_real_eyes/ckpt/midday run_checkpoint.model.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 95s 2s/step - accuracy: 0.7564 - loss: 7.2508 - val_accuracy: 0.8710 - val_loss: 6.9579 - learning_rate: 1.0000e-05
Epoch 3/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8114 - loss: 7.1090
Epoch 3: val_accuracy improv

In [ ]:
# save the history
history_df = pd.DataFrame(history.history)
history_df['epoch'] = history.epoch
history_df.to_csv(history_file_path, index=False)

In [10]:
# load the best model
model.load_weights(ckpt_file_path)

In [11]:
# save the model
model.save(model_file_path)

In [ ]:
plot_history(comment=comment, history=history)

In [12]:
evaluate_model(model, real_test_data)

16/16 ━━━━━━━━━━━━━━━━━━━━ 10s 598ms/step - accuracy: 0.9444 - loss: 6.7896
Test loss: 6.754405498504639
Test accuracy: 0.9577319622039795
